In [2]:
import sys
import pathlib
import math
import numpy as np
import torch
import torch.nn.functional as F
import scipy.stats as stats
from hadamard_transform import randomized_hadamard_transform, inverse_randomized_hadamard_transform, pad_to_power_of_2

PROJECT_PATH = pathlib.Path.cwd().parent
if str(PROJECT_PATH) not in sys.path:
    sys.path.append(str(PROJECT_PATH))

from pcdvq import (
    e8_minimal_directions,
    construct_direction_codebook,
    construct_magnitude_codebook,
    PCDVQ,
)
from pcdvq.utils import reshape_pq_to_k, reshape_k_to_pq
from pcdvq.standard_regularization import RandomizedHadamard

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
def e8_roots(device=None, dtype=torch.float16, normalize=False) -> torch.Tensor:
    eye_matrix = torch.eye(8, dtype=dtype, device=device)
    row_idxs, col_idxs = torch.triu_indices(8, 8, offset=1, dtype=torch.int8, device=device)
    e1, e2 = eye_matrix[row_idxs], eye_matrix[col_idxs]
    int_roots = torch.cat([e1 + e2, e1 - e2, -e1 + e2, -e1 - e2], 0)

    signs = torch.tensor([-1.0, 1.0], dtype=dtype, device=device)
    halfs = torch.cartesian_prod(*([signs] * 8)).reshape(-1, 8)
    half_roots = 0.5 * [halfs.prod(dim=-1) > 0]
    
    e8_roots = torch.cat([int_roots, half_roots], dim=0)    
    if normalize:
        e8_roots = e8_roots / math.sqrt(2.0)
    return e8_roots

In [ ]:


S.shape

torch.Size([128, 8])

In [2]:
tensors_dumps_dir = "tensors_dumps"
weight_filename = "weight.pt"
weight_path = PROJECT_PATH / tensors_dumps_dir / weight_filename
weight = torch.load(weight_path)
p, q  = weight.shape
weight

tensor([[-2.1744e-04, -1.0193e-02,  9.5825e-03,  ..., -3.1738e-03,
         -8.6670e-03,  1.3504e-03],
        [ 2.8076e-02, -2.3804e-03, -1.2634e-02,  ..., -1.9989e-03,
          1.9684e-03,  1.5503e-02],
        [-1.6357e-02,  5.8289e-03, -1.7456e-02,  ..., -1.1414e-02,
          1.6174e-03,  2.8372e-05],
        ...,
        [ 2.2949e-02, -1.2024e-02, -2.3438e-02,  ..., -2.0752e-02,
          3.9795e-02,  2.7588e-02],
        [-2.9663e-02,  2.0142e-02,  1.4465e-02,  ..., -2.7222e-02,
         -6.2561e-03, -3.8818e-02],
        [ 1.1169e-02,  4.4922e-02, -7.2937e-03,  ...,  3.0518e-02,
         -2.3438e-02, -2.3071e-02]], dtype=torch.float16)

In [4]:
phi_bits = 16
r_bits = 2
k = 8
tau = 0.15
tol = 1e-5
iters = 100
dtype = torch.float16
device = 'cpu'

C_phi = construct_direction_codebook(phi_bits, dtype=dtype, device=device)
C_r = construct_magnitude_codebook(r_bits, k, tau, tol, iters)
pcdvq = PCDVQ(directions_codebook=C_phi, magnitudes_codebook=C_r)

In [5]:
C_phi

tensor([[ 0.3535,  0.3535, -0.3535,  ...,  0.3535,  0.3535,  0.3535],
        [-0.3535, -0.3535,  0.3535,  ..., -0.3535, -0.3535, -0.3535],
        [ 0.7070,  0.0000,  0.7070,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [ 0.7070,  0.7070,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.7070,  0.7070,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.7070,  0.7070,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       dtype=torch.float16)

In [6]:
C_r

tensor([0.2524, 1.2513, 1.6237, 1.9007])

In [8]:
weight_reshaped = reshape_pq_to_k(weight, k)
pcdvq_res = pcdvq.forward(weight_reshaped)
weight_reshaped_quant = pcdvq_res["x_q"]
weight_quant = reshape_k_to_pq(weight_reshaped_quant, p, q)
mse_val = F.mse_loss(weight.float(), weight_quant.float()).item()
mse_val

RuntimeError: [enforce fail at alloc_cpu.cpp:117] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 150323855360 bytes. Error code 12 (Cannot allocate memory)

In [9]:
weight_quant

NameError: name 'weight_quant' is not defined

In [10]:
pcdvq_res

NameError: name 'pcdvq_res' is not defined

In [11]:
prng = torch.Generator(device='cpu')
prng.manual_seed(42)
state = prng.get_state()
p, q = weight_reshaped.shape
weight_sgr = randomized_hadamard_transform(pad_to_power_of_2(weight_reshaped.T), prng=prng).T
weight_sgr, weight_sgr.shape, p, q

(tensor([[-0.0269, -0.0089,  0.0016,  ...,  0.0390,  0.0090,  0.0102],
         [-0.0186,  0.0009, -0.0147,  ..., -0.0008, -0.0114, -0.0030],
         [ 0.0112,  0.0110,  0.0145,  ...,  0.0035,  0.0328,  0.0279],
         ...,
         [-0.0027,  0.0309, -0.0396,  ..., -0.0012, -0.0175, -0.0044],
         [ 0.0064, -0.0124, -0.0174,  ...,  0.0216,  0.0006,  0.0068],
         [ 0.0149, -0.0102,  0.0106,  ..., -0.0181,  0.0082, -0.0233]],
        dtype=torch.float16),
 torch.Size([2097152, 8]),
 1310720,
 8)

In [12]:
prng.set_state(state)
weight_sgr_reverse = inverse_randomized_hadamard_transform(weight_sgr.T, prng=prng).T[:p]
weight_sgr_reverse, weight_sgr_reverse.shape

(tensor([[-0.0002, -0.0102,  0.0096,  ...,  0.0099, -0.0084, -0.0004],
         [-0.0039, -0.0015,  0.0006,  ..., -0.0055, -0.0152, -0.0031],
         [-0.0078, -0.0131, -0.0086,  ...,  0.0030,  0.0048,  0.0047],
         ...,
         [-0.0183,  0.0081, -0.0043,  ..., -0.0229, -0.0159,  0.0220],
         [-0.0033,  0.0273, -0.0347,  ...,  0.0073,  0.0064,  0.0292],
         [ 0.0021,  0.0155,  0.0420,  ...,  0.0305, -0.0235, -0.0231]],
        dtype=torch.float16),
 torch.Size([1310720, 8]))

In [13]:
weight_reshaped

tensor([[-0.0002, -0.0102,  0.0096,  ...,  0.0099, -0.0084, -0.0005],
        [-0.0038, -0.0014,  0.0006,  ..., -0.0055, -0.0151, -0.0031],
        [-0.0078, -0.0131, -0.0086,  ...,  0.0030,  0.0048,  0.0047],
        ...,
        [-0.0183,  0.0081, -0.0042,  ..., -0.0228, -0.0159,  0.0220],
        [-0.0033,  0.0273, -0.0347,  ...,  0.0074,  0.0064,  0.0292],
        [ 0.0021,  0.0154,  0.0420,  ...,  0.0305, -0.0234, -0.0231]],
       dtype=torch.float16)

In [14]:
assert torch.allclose(weight_reshaped, weight_sgr_reverse, atol=1e-3)

In [17]:
phi, r = PCDVQ.to_polar(weight_sgr)
reshaped_phis = reshape_pq_to_k(phi, 8)
reshaped_phis

tensor([[1.9189, 1.6904, 1.5488,  ..., 0.3359, 0.8438, 2.1387],
        [1.5410, 2.1016, 2.4824,  ..., 3.3984, 1.3457, 1.3447],
        [1.2627, 1.7129, 1.2832,  ..., 1.7471, 1.8135, 1.6719],
        ...,
        [0.4575, 2.6543, 2.2031,  ..., 2.6387, 1.8613, 1.0479],
        [1.6387, 3.3848, 1.4707,  ..., 0.4026, 1.8379, 0.3037],
        [1.4805, 1.1982, 1.8418,  ..., 1.3340, 2.2051, 5.0469]],
       dtype=torch.float16)

In [20]:
reshaped_phis.shape

torch.Size([1835008, 8])

In [21]:
def argmax_cosine_chunked(phis, C_phi, chunk=65536, eps=1e-12,
                          device=None, dtype=None):
    if device is None:
        device = phis.device
    if dtype is None:
        dtype = phis.dtype

    z = phis.to(device=device, dtype=dtype)
    z = z / (z.norm(dim=-1, keepdim=True).clamp_min(eps))

    Z = C_phi.to(device=device, dtype=dtype)
    Z = Z / (Z.norm(dim=-1, keepdim=True).clamp_min(eps))

    N, M = z.size(0), Z.size(0)
    idx = torch.empty(N, dtype=torch.long, device=device)
    vals = torch.empty(N, dtype=dtype, device=device)

    for i in range(0, N, chunk):
        zz = z[i:i+chunk]
        sim_b = zz @ Z.t()
        v, j = sim_b.max(dim=1)
        idx[i:i+chunk] = j
        vals[i:i+chunk] = v

    return idx, vals

In [ ]:
# z = torch.nn.functional.normalize(reshaped_phis, dim=-1)
# Z = torch.nn.functional.normalize(C_phi.to(z.dtype), dim=-1)
# sim = z @ Z.T
# idx_dir = sim.argmax(dim=1)
sim = argmax_cosine_chunked(reshaped_phis, C_phi.to(reshaped_phis.dtype))
sim

In [218]:
idx_dir

tensor([26, 54, 58,  ..., 38, 48, 32])

In [219]:
d = (r.view(-1, 1) - C_r.view(1, -1)).abs()
idx_rad = d.argmin(dim=1)
idx_rad

tensor([1, 1, 1,  ..., 1, 1, 1])

In [220]:
r

tensor([0.1486, 0.1501, 0.1587,  ..., 0.1617, 0.1320, 0.1486],
       dtype=torch.float16)

In [221]:
phis_q = Z[idx_dir]
phi_p, phi_q = phi.shape
reshaped_phis_q = reshape_k_to_pq(phis_q, phi_p, phi_q)
r_q = C_r[idx_rad].unsqueeze(1)
prng.set_state(state)
weight_q = inverse_randomized_hadamard_transform(PCDVQ.to_cartesian(reshaped_phis_q, r_q).T, prng=prng).T[:p]

In [222]:
weight_q

tensor([[-5.5400e+00,  3.9788e-01,  3.8280e-02,  ...,  0.0000e+00,
         -0.0000e+00, -0.0000e+00],
        [ 1.8272e-02, -1.3904e-01, -4.0908e-02,  ...,  0.0000e+00,
         -0.0000e+00, -0.0000e+00],
        [-1.0417e-02, -5.3465e-02,  2.1540e-02,  ..., -0.0000e+00,
         -0.0000e+00,  0.0000e+00],
        ...,
        [-2.8018e-04, -1.7252e-03,  7.1873e-04,  ...,  0.0000e+00,
         -0.0000e+00,  0.0000e+00],
        [-7.6397e-04, -3.3922e-03,  4.5999e-05,  ...,  0.0000e+00,
          0.0000e+00, -0.0000e+00],
        [ 9.5730e-04, -2.6090e-03, -8.7808e-04,  ...,  0.0000e+00,
          0.0000e+00, -0.0000e+00]])

In [223]:
phis_q, r_q

(tensor([[0.0000, 0.7070, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.7070, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.7070, 0.7070, 0.0000],
         ...,
         [0.0000, 0.0000, 0.7070,  ..., 0.7070, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.7070, 0.0000],
         [0.0000, 0.7070, 0.0000,  ..., 0.0000, 0.0000, 0.7070]],
        dtype=torch.float16),
 tensor([[0.0111],
         [0.0111],
         [0.0111],
         ...,
         [0.0111],
         [0.0111],
         [0.0111]]))

In [179]:
phis_q[5]

tensor([0., 1., 0., 0., 0., 1., 0., 0.], dtype=torch.float16)

In [226]:
# C — кодбук (K,8), qdir — квантизированные направления (...,8), idx — индексы (из квантизатора)

# 1) Проверка соответствия: qdir == C[idx]
ok = (C_phi[idx_dir.reshape(-1)] - phis_q.reshape(-1, 8)).abs().max().item() < 1e-6
print("qdir equals codebook rows:", ok)

# 2) Есть ли отрицательные значения и тип-B (все 8 компонент ≈ 0.3536 по модулю)?
print("qdir has negatives:", (phis_q < 0).any().item())
abs_vals = torch.unique(phis_q.abs().flatten().round(decimals=4))
print("unique |values| in qdir:", abs_vals.tolist())

# тип-B присутствует, если есть блоки, где все 8 компонент по модулю ~0.3536
typeB_in_qdir = (((phis_q.abs() > 0.30) & (phis_q.abs() < 0.40)).sum(dim=-1) == 8).any().item()
print("type-B present in qdir:", typeB_in_qdir)


qdir equals codebook rows: True
qdir has negatives: True
unique |values| in qdir: [0.0, 0.353515625, 0.70703125]
type-B present in qdir: True


In [227]:
x = reshaped_phis
G = x.size(-1)//8
r = torch.linalg.vector_norm(x.reshape(-1, G, 8).float(), dim=-1)
print("r min/median/max:", r.min().item(), r.median().item(), r.max().item())


r min/median/max: 2.576680898666382 4.491796016693115 8.53427505493164


In [228]:
xg = x.reshape(-1, G, 8).float()
u  = xg / (torch.linalg.vector_norm(xg, dim=-1, keepdim=True).clamp_min(1e-12))
cos_mean = (u.reshape(-1,8) * phis_q.reshape(-1,8)).sum(-1).mean().item()
print("mean cosine(u, qdir):", round(cos_mean, 4))


mean cosine(u, qdir): 0.5858
